# PhishGuard — SMS Spam Model Generator

In this notebook, we document how we generate the SMS spam-classification model and its TF-IDF vectorizer.

## Outputs
- `model.pkl` — trained soft-voting classifier
- `vectorizer.pkl` — fitted TF-IDF vectorizer

## Requirements
Run this notebook from the project directory containing `spam.csv`, or update `DATASET_PATH` below. The saved pickle files are written to the current working directory.

We preserve the core preprocessing, vectorization, train/test split, classifier configuration, and model-saving approach used in our original project.

## 1. Import libraries

In [ ]:
import pickle
import string

import nltk
import numpy as np
import pandas as pd

from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer

from sklearn.ensemble import ExtraTreesClassifier, VotingClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, confusion_matrix, precision_score
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import SVC

## 2. Load and clean the SMS dataset

We use `v1` for the label and `v2` for the message. We encode the labels as `0` for ham (legitimate) and `1` for spam.

In [ ]:
DATASET_PATH = "spam.csv"

df = pd.read_csv(DATASET_PATH, encoding="latin-1")

# Remove unused dataset columns and give the useful columns clear names.
unused_columns = ["Unnamed: 2", "Unnamed: 3", "Unnamed: 4"]
df.drop(columns=unused_columns, inplace=True)
df.rename(columns={"v1": "target", "v2": "text"}, inplace=True)

# Encode ham/spam labels: ham -> 0, spam -> 1.
df["target"] = df["target"].map({"ham": 0, "spam": 1})

# Remove duplicate messages, keeping the first occurrence.
df = df.drop_duplicates(keep="first").reset_index(drop=True)

if df["target"].isna().any():
    raise ValueError("Unexpected label found. Expected only 'ham' and 'spam'.")

print(f"Dataset shape after cleaning: {df.shape}")
print(df["target"].value_counts().rename(index={0: "ham", 1: "spam"}))

## 3. Prepare text preprocessing

We lowercase each message, tokenize it, keep alphanumeric tokens, remove English stopwords and punctuation, and apply Porter stemming. We use the same text transformation when preparing messages for prediction.

In [ ]:
# Download the NLTK resources needed by tokenization and stopword removal.
nltk.download("punkt")
nltk.download("stopwords")

# Some NLTK versions require this tokenizer resource as well.
nltk.download("punkt_tab", quiet=True)

porter = PorterStemmer()
english_stopwords = set(stopwords.words("english"))

def transform_text(text):
    text = str(text).lower()
    tokens = nltk.word_tokenize(text)

    tokens = [token for token in tokens if token.isalnum()]
    tokens = [
        token for token in tokens
        if token not in english_stopwords and token not in string.punctuation
    ]
    tokens = [porter.stem(token) for token in tokens]

    return " ".join(tokens)

df["transformed_text"] = df["text"].apply(transform_text)

df[["target", "text", "transformed_text"]].head()

## 4. Convert messages into TF-IDF features

In our project, we use a TF-IDF vectorizer limited to 3,000 features. We retain this configuration so our generated vectorizer remains consistent with the application's expected model input.

In [ ]:
tfidf = TfidfVectorizer(max_features=3000)
X = tfidf.fit_transform(df["transformed_text"]).toarray()
y = df["target"].values

print(f"Feature matrix shape: {X.shape}")

## 5. Split the dataset

We split our dataset into 80% training data and 20% testing data, using `random_state=2` so we can reproduce the same split.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=2
)

print(f"Training samples: {len(X_train)}")
print(f"Testing samples: {len(X_test)}")

## 6. Build and evaluate the Voting Classifier

We combine an SVC, Multinomial Naive Bayes, and Extra Trees classifier using soft voting. We keep the estimator settings consistent with our original model-generation code.

In [ ]:
svc = SVC(kernel="sigmoid", gamma=1.0, probability=True)
mnb = MultinomialNB()
etc = ExtraTreesClassifier(n_estimators=50, random_state=2)

voting = VotingClassifier(
    estimators=[("svm", svc), ("nb", mnb), ("et", etc)],
    voting="soft"
)

voting.fit(X_train, y_train)
y_pred = voting.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Confusion matrix:")
print(confusion_matrix(y_test, y_pred))

## 7. Save the trained artifacts

We save both files in the project directory where our Django application expects to load them. We keep the filenames unchanged.

In [ ]:
with open("../vectorizer.pkl", "wb") as file:
    pickle.dump(tfidf, file)

with open("../model.pkl", "wb") as file:
    pickle.dump(voting, file)

print("Model and TF-IDF vectorizer saved successfully:")
print("- model.pkl")
print("- vectorizer.pkl")

## 8. Reproducibility notes

- We keep the dataset available at `DATASET_PATH` whenever we rerun this notebook.
- We save the two pickle files to the current working directory.
- Our application uses preprocessing compatible with `transform_text` above and the fitted TF-IDF vectorizer.
- We load pickle files only from trusted sources because pickle files can execute code when loaded.
